# 00 — Setup & Environment Check

Welcome to **learningdantic**. This repo teaches you [Pydantic AI v2](https://pydantic.dev/docs/ai/)
and [Logfire](https://pydantic.dev/logfire) the only way that sticks: by running real code yourself.

How it works:

- Run every notebook **top to bottom**, cell by cell — don't skip the "you should see X" checkpoints.
- Notebooks **01+** end with an **exercise**. Try it alone first, then compare with the matching notebook
  in [`../solutions/`](../solutions/).
- This first notebook needs **no API key at all** — it verifies your environment with pydantic-ai's
  built-in fake model, so nothing can go wrong expensively.

## What you need before this notebook

- [x] `uv sync` — run once from the repo root (creates `.venv`, installs everything, Python included)
- [x] `cp .env.example .env` — created your `.env` file (a key is optional *for this notebook*)
- [ ] Put your `OPENAI_API_KEY` into `.env` — needed from notebook 01 onwards
- [ ] *Optional:* `uv run logfire auth` in a terminal — browser login to the free Logfire tier
  (10M spans/month, no credit card) so you can watch traces at [logfire.pydantic.dev](https://logfire.pydantic.dev)

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

> ✅ **Checkpoint** — you should see:
>
> ```
> kernel : /home/.../learningdantic/.venv/bin/python
> MODEL  : openai:gpt-6.1-sol
> ```
>
> If `kernel:` points somewhere else (like `/usr/bin/python3`), your notebook is running the wrong
> environment: restart Jupyter with `uv run jupyter lab` from the repo root, or use
> *Kernel → Change Kernel* in Jupyter. A wrong kernel is the #1 cause of `ModuleNotFoundError`.

## The asyncio ground rules

Jupyter runs an asyncio event loop behind every cell, and pydantic-ai is an async-first framework.
Five rules keep you out of trouble — they apply to **every** notebook in this repo:

1. **Top-level `await` just works** in modern Jupyter: `result = await agent.run('...')`.
   No `nest_asyncio` (that's for legacy setups, Colab, or marimo — not here).
2. **`run_sync()` only at the top level of a notebook cell.** Never inside a tool, an output
   function, or any `async def` — sync code runs on worker threads, and a nested `run_sync`
   **deadlocks silently** (sometimes it even "works" on the first call, then hangs forever).
3. **Tools and output functions are always `async def`** — and if they delegate to another
   agent, they use `await other_agent.run(...)`.
4. **`RuntimeError: Event loop is closed`** → restart the kernel and recreate your agent/model.
   A dead loop takes the model's connection pool with it.
5. **Logfire is configured once per kernel** — re-running the setup cell after a kernel restart
   is fine and expected.

In [ ]:
import asyncio

print("An event loop is running right now:", asyncio.get_running_loop() is not None)
print("Top-level await works:", await asyncio.sleep(0, result=True))

> ✅ **Checkpoint** — the loop is running and top-level `await` returned `True`.
> That's all rule 1 needs: from now on, you can `await` any agent call directly in a cell.

## Keyless smoke test — TestModel

Before spending a single token, verify the whole stack with pydantic-ai's built-in **fake model**.
`TestModel` never touches the network — it's how the repo's CI runs notebooks without any API keys.

(Side benefit: if you ever see a confusing `UserError` telling you to "use the built-in test model",
this is what it means.)

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.test import TestModel

smoke_agent = Agent(TestModel(), instructions="You are a smoke test.")

smoke_result = await smoke_agent.run("Is anybody out there?")
print("output :", repr(smoke_result.output))
print("usage  :", smoke_result.usage)  # RunUsage — a *property* in v2, not a method

> ✅ **Checkpoint** — you should see something like:
>
> ```
> output : 'success (no tool calls)'
> usage  : RunUsage(input_tokens=54, output_tokens=4, requests=1)
> ```
>
> Pydantic AI v2 is installed, imports cleanly, and runs agents in your kernel.
> `RunUsage` counts requests so you can audit cost — with a real model it also fills in `cost` (USD).

## Logfire check

The setup cell already ran the two magic lines:

```python
logfire.configure(send_to_logfire='if-token-present')
logfire.instrument_pydantic_ai()
```

That's the **entire** integration — in v2 instrumentation is always explicit. Because of
`if-token-present`, everything works without an account and automatically lights up when you run
`logfire auth` or set `LOGFIRE_TOKEN`.

In [ ]:
with logfire.span("notebook_00_smoke"):
    logfire.info("learningdantic is alive: setup, asyncio and TestModel all work")

print("Ran `logfire auth`? Open the Live view at https://logfire.pydantic.dev")
print("and find the notebook_00_smoke span above.")
print("No account yet? Everything above still ran — Logfire is optional, not required.")

## Troubleshooting

| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: pydantic_ai` | Wrong kernel — check `sys.executable` points into `learningdantic/.venv`, restart via `uv run jupyter lab` |
| A cell hangs forever | You called `run_sync()` inside a tool/async function — silent deadlock. Restart kernel, use `async def` + `await` |
| `RuntimeError: Event loop is closed` | Restart the kernel; recreate your agent objects |
| Nothing appears in the Logfire UI | Run `uv run logfire auth`, confirm a `.logfire/` dir exists (or set `LOGFIRE_TOKEN`) |
| `UserError` about model names | v2 requires prefixed strings: `'openai:gpt-6.1-sol'`, never bare `'gpt-6.1-sol'` |

## Next

→ **[01_first_agent.ipynb](01_first_agent.ipynb)** — your first real, live, paying-actual-tokens agent.
Make sure your `.env` has `OPENAI_API_KEY=...` first.